# Forecasting de Variables

Usamos ScikitLearn para estimar los valores futuros (1 ano) de las variables de 'datos sinteticos'; usamos los propios datos generados para llevarlo a cabo. 

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.tsa.holtwinters import ExponentialSmoothing

#  CARGA DE DATOS SINTÉTICOS HISTÓRICOS (2010 - 2026)

data_dir = Path('datos_sinteticos')

df_clima_hist = pd.read_csv(data_dir / 'dim_clima.csv')
df_mercado_hist = pd.read_csv(data_dir / 'dim_mercado.csv')
df_lotes = pd.read_csv(data_dir / 'dim_lotes.csv')

# FORECAST A 1 AÑO (52 SEMANAS) DE CLIMA Y PRECIOS

# A. Forecast de Lluvia y Temperatura
model_lluvia = ExponentialSmoothing(
    df_clima_hist['lluvia_semanal_mm'],
    trend='add',
    seasonal='add',
    seasonal_periods=52,
    initialization_method='heuristic'
).fit(
    method='L-BFGS-B'
)

model_temp = ExponentialSmoothing(
    df_clima_hist['temperatura_media_c'],
    trend='add',
    seasonal='add',
    seasonal_periods=52,
    initialization_method='heuristic'
).fit(
    method='L-BFGS-B'
)
lluvia_forecast = model_lluvia.forecast(52)
temp_forecast = model_temp.forecast(52)

# B. Forecast de Precios de Insumos y Precio del Limón 
model_precio_urea = ExponentialSmoothing(
    df_mercado_hist['precio_urea_usd_kg'], trend='add'
).fit()
model_precio_fungicida = ExponentialSmoothing(
    df_mercado_hist['precio_fungicida_usd_l'], trend='add'
).fit()
model_precio_gasoil = ExponentialSmoothing(
    df_mercado_hist['precio_gasoil_usd_l'], trend='add'
).fit()

col_exp = [c for c in df_mercado_hist.columns if 'exp' in c.lower() or 'export' in c.lower()]
col_ind = [c for c in df_mercado_hist.columns if 'ind' in c.lower() or 'industria' in c.lower()]

col_exp_nombre = col_exp[0] if col_exp else 'precio_limon_exportacion_usd_t'
col_ind_nombre = col_ind[0] if col_ind else 'precio_limon_industria_usd_t'

model_precio_exp = ExponentialSmoothing(
    df_mercado_hist[col_exp_nombre],
    trend='add',
    seasonal='add',
    seasonal_periods=52,
    initialization_method='heuristic'
).fit(method='L-BFGS-B')

model_precio_ind = ExponentialSmoothing(
    df_mercado_hist[col_ind_nombre],
    trend='add',
    seasonal='add',
    seasonal_periods=52,
    initialization_method='heuristic'
).fit(method='L-BFGS-B')

precio_urea_forecast = model_precio_urea.forecast(52)
precio_fungicida_forecast = model_precio_fungicida.forecast(52)
precio_gasoil_forecast = model_precio_gasoil.forecast(52)
precio_limon_exp_forecast = model_precio_exp.forecast(52)
precio_limon_ind_forecast = model_precio_ind.forecast(52)

# Generación del DataFrame de Proyección Futura 2026-2027
N_PERIODOS = 52
fechas_futuras = pd.date_range(start='2026-08-10', periods=N_PERIODOS, freq='W-MON')

# Forzar a que los forecasts devuelvan exactamente 52 períodos
lluvia_pred = np.clip(model_lluvia.forecast(N_PERIODOS).values, 0, None).round(1)
temp_pred = model_temp.forecast(N_PERIODOS).values.round(1)

precio_urea_pred = model_precio_urea.forecast(N_PERIODOS).values.round(2)
precio_fungicida_pred = model_precio_fungicida.forecast(N_PERIODOS).values.round(2)
precio_gasoil_pred = model_precio_gasoil.forecast(N_PERIODOS).values.round(2)

precio_exp_pred = np.clip(model_precio_exp.forecast(N_PERIODOS).values, 0, None).round(2)
precio_ind_pred = np.clip(model_precio_ind.forecast(N_PERIODOS).values, 0, None).round(2)

# 3. Ensamblar el DataFrame con las proyecciones de ambos mercados
df_forecast_variables = pd.DataFrame({
    'fecha': fechas_futuras.strftime('%Y-%m-%d'),
    'mes': fechas_futuras.month,
    'semana_anio': fechas_futuras.isocalendar().week.astype(int),
    'lluvia_semanal_mm_pred': lluvia_pred,
    'temperatura_media_c_pred': temp_pred,
    'precio_urea_usd_kg_pred': precio_urea_pred,
    'precio_fungicida_usd_l_pred': precio_fungicida_pred,
    'precio_gasoil_usd_l_pred': precio_gasoil_pred,
    'precio_limon_exportacion_usd_t_pred': precio_exp_pred,
    'precio_limon_industria_usd_t_pred': precio_ind_pred
})

# PROYECCIÓN DE DEMANDA DE INSUMOS 

registros_proyectados = []

for _, lote in df_lotes.iterrows():
    for _, fila in df_forecast_variables.iterrows():
        mes = fila['mes']
        lluvia = fila['lluvia_semanal_mm_pred']
        temp = fila['temperatura_media_c_pred']
        
        # Regla Urea: Primavera / Verano temprano (Septiembre a Diciembre)
        if mes in [9, 10, 11, 12]:
            demanda_urea_kg_ha = round(np.random.normal(40, 5), 1)
        else:
            demanda_urea_kg_ha = 0.0
            
        # Regla Fungicida: Alta humedad (lluvia > 20 mm) en meses calidos
        if lluvia > 20.0 and mes in [11, 12, 1, 2, 3]:
            demanda_fungicida_l_ha = round(np.random.normal(2.5, 0.4), 1)
        else:
            demanda_fungicida_l_ha = 0.0
            
        # Regla Gasoil: Base constante + extra por aplicaciones
        gasoil_base = 6.0 if mes in [4, 5, 6, 7, 8] else 3.0
        if demanda_fungicida_l_ha > 0 or demanda_urea_kg_ha > 0:
            gasoil_base += 2.5
        demanda_gasoil_l_ha = round(gasoil_base + np.random.normal(0, 0.5), 1)
        
        # Cálculo monetario
        costo_urea = demanda_urea_kg_ha * fila['precio_urea_usd_kg_pred']
        costo_fungicida = demanda_fungicida_l_ha * fila['precio_fungicida_usd_l_pred']
        costo_gasoil = demanda_gasoil_l_ha * fila['precio_gasoil_usd_l_pred']
        costo_total_ha = costo_urea + costo_fungicida + costo_gasoil
        
        registros_proyectados.append({
            'fecha': fila['fecha'],
            'lote_id': lote['lote_id'],
            'nombre_finca': lote['nombre_finca'],
            'demanda_urea_kg_ha': demanda_urea_kg_ha,
            'demanda_fungicida_l_ha': demanda_fungicida_l_ha,
            'demanda_gasoil_l_ha': demanda_gasoil_l_ha,
            'costo_insumos_usd_ha': round(costo_total_ha, 2),
            'costo_total_lote_usd': round(costo_total_ha * lote['superficie_ha'], 2)
        })

df_proyeccion_2027 = pd.DataFrame(registros_proyectados)

# Exportación de la proyección final a la carpeta de datos
df_proyeccion_2027.to_csv(data_dir / 'hecho_proyeccion_2027.csv', index=False)
df_forecast_variables.to_csv(data_dir / 'dim_forecast_clima_precios_2027.csv', index=False)

print("--- FORECAST A 1 AÑO Y PROYECCIÓN DE DEMANDA COMPLETADOS ---")


# FORECAST DE RINDE Y PRODUCCIÓN 

# Agrupar las variables climáticas proyectadas para el año
lluvia_total_2027 = df_forecast_variables['lluvia_semanal_mm_pred'].sum()

# Identificar semanas con riesgo de helada en la proyección (temp < 5 °C en invierno)
heladas_2027 = df_forecast_variables[
    (df_forecast_variables['mes'].isin([6, 7])) & 
    (df_forecast_variables['temperatura_media_c_pred'] < 5.0)
].shape[0]

# Determinar el factor de impacto climático proyectado
factor_clima_2027 = 1.0

if lluvia_total_2027 < 700:
    factor_clima_2027 -= 0.25
elif lluvia_total_2027 < 900:
    factor_clima_2027 -= 0.10

if heladas_2027 >= 3:
    factor_clima_2027 -= 0.30
elif heladas_2027 >= 1:
    factor_clima_2027 -= 0.15

factor_clima_2027 = max(0.4, factor_clima_2027)

# Proyectar el rinde real y la producción total por lote
proyeccion_rindes_lotes = []

for _, lote in df_lotes.iterrows():
    # Rinde proyectado aplicando el factor climático + variabilidad normal
    rinde_proyectado = (lote['rinde_historico_t_ha'] * factor_clima_2027) + np.random.normal(0, 1.0)
    rinde_final = round(max(10.0, rinde_proyectado), 1)
    
    proyeccion_rindes_lotes.append({
        'anio_campana': '2026-2027',
        'lote_id': lote['lote_id'],
        'nombre_finca': lote['nombre_finca'],
        'variedad': lote['variedad'],
        'superficie_ha': lote['superficie_ha'],
        'factor_clima_proyectado': round(factor_clima_2027, 2),
        'rinde_proyectado_t_ha': rinde_final,
        'produccion_proyectada_t': round(rinde_final * lote['superficie_ha'], 1)
    })

df_proyeccion_rindes_2027 = pd.DataFrame(proyeccion_rindes_lotes)

# Exportar tabla de rindes proyectados
df_proyeccion_rindes_2027.to_csv(data_dir / 'hecho_proyeccion_rindes_2027.csv', index=False)

--- FORECAST A 1 AÑO Y PROYECCIÓN DE DEMANDA COMPLETADOS ---
